<a href="https://colab.research.google.com/github/RLeidyFerreira/Criacao-notebookLM/blob/main/Assistente_de_Voz_Mult_idiomas_com_whisper_e_ChatGPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
language = "pt"

1- Gravação de Audio com o Python (e Uma Pitada de JavaScript)

In [19]:
import base64
from IPython.display import display, Javascript, Audio
from google.colab import output

# JavaScript code to record audio
RECORD = """
  const sleep = time => new Promise(resolve => setTimeout(resolve, time));
  const b2text = blob => new Promise(resolve => {
    const reader = new FileReader();
    reader.onloadend = () => resolve(reader.result);
    reader.readAsDataURL(blob);
  });
  window.record = async (time) => { // Explicitly define on window object
    const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
    const recorder = new MediaRecorder(stream);
    const data = [];
    recorder.ondataavailable = e => data.push(e.data);
    recorder.start();
    await sleep(time);
    recorder.stop();
    await sleep(200);
    stream.getTracks().forEach(track => track.stop());
    return await b2text(new Blob(data));
  };
"""

# Display the Javascript code once to define the 'record' function in the browser's scope
display(Javascript(RECORD))

def record(sec=1):
    # Now, the JavaScript 'record' function should be defined
    js_result = output.eval_js('record(%s)' % (sec * 1000))
    audio = base64.b64decode(js_result.split(',')[1])
    file_name = 'request_audio.wav'
    with open(file_name, 'wb') as f:
        f.write(audio)
    return f'/content/{file_name}'

print('Ouvindo...\n')
record_file = record()
display(Audio(record_file, autoplay=True))

<IPython.core.display.Javascript object>

Ouvindo...



In [20]:
from gtts import gTTS
from IPython.display import Audio

test_text = "Este é um áudio de teste para o seu sistema."
test_audio_file = "/content/test_audio.wav"

gtts_object_test = gTTS(text=test_text, lang='pt', slow=False)
gtts_object_test.save(test_audio_file)

print(f"Áudio de teste gerado em: {test_audio_file}")
display(Audio(test_audio_file, autoplay=True))

Áudio de teste gerado em: /content/test_audio.wav


Agora você pode usar `test_audio_file` na célula de transcrição do Whisper. Lembre-se de substituir `record_file` por `test_audio_file` na célula `HVFhD8vLygSs` (ou em qualquer célula que chame o Whisper).

2-Reconhecimento de fala com Whisper(OpenAI)

In [21]:
!pip install git+https://github.com/openai/whisper.git -q


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


3-Integração com a API do ChatGPT

In [22]:
import whisper

model = whisper.load_model("small")
result = model.transcribe(test_audio_file, fp16=False, language = language)
transcription = result["text"]
print(transcription)

 Este é um áudio de teste para o seu sistema.


3-Integração com a API do ChatGPT

In [23]:
!pip install openai

In [24]:
# Importe o SDK do Python
import google.generativeai as genai
# Usado para armazenar sua chave de API com segurança
from google.colab import userdata

GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

Agora, vamos inicializar o modelo Gemini e usá-lo para gerar uma resposta com base na sua transcrição:

In [25]:
# Inicialize o modelo Gemini
gemini_model = genai.GenerativeModel('gemini-flash-latest') # Substituindo 'gemini-pro' por um modelo válido

# Faça a chamada para a API do Gemini com a transcrição
try:
    response = gemini_model.generate_content(transcription)
    gemini_response = response.text
    print(gemini_response)
except Exception as e:
    print(f"Ocorreu um erro ao chamar a API do Gemini: {e}")
    print("Listando modelos disponíveis para depuração...")
    for m in genai.list_models():
        if "generateContent" in m.supported_generation_methods:
            print(m.name)

Recebido em alto e bom som! O teste foi bem-sucedido.

Como posso ajudar você hoje?


4- Sintetizar a Resposta do ChatGPT com Voz(gTTS)

In [26]:
!pip install gTTS

In [27]:
from gtts import gTTS
gtts_object = gTTS(text=gemini_response, lang= language, slow=False)
response_audio= "/content/response_audio.wav"
gtts_object.save(response_audio)
display(Audio(response_audio,autoplay=True))